<B>Key Components</B> \
Language Model: The core AI component that generates responses. \
Prompt Template: Defines the structure of our conversations. \
History Manager: Manages conversation history and context. \
Message Store: Stores the messages for each conversation session. 

<B>Method Details</B>
1. Setting ENv
2. Creating the Chat History Store
3. Defining the Conversation Structure
4. Building the Conversational Chain
5. Interacting with the Agent
6. 

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import messages_from_dict, messages_to_dict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import os

In [2]:
"""
TUTORIAL 1: Simple Conversational Agent
Using existing Gen-AI project structure
"""

import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent  # Go up from notebooks/ to Gen-AI/
sys.path.insert(0, str(project_root))

# Use your existing env_loader
from utils.env_loader import load_environment

# Load environment
load_environment()

print("="*80)
print("TUTORIAL 1: SIMPLE CONVERSATIONAL AGENT")
print("="*80)
print(f"Project root: {project_root}")
print(f"Working from: {Path.cwd()}")

TUTORIAL 1: SIMPLE CONVERSATIONAL AGENT
Project root: /Users/kanderaolaxminarasimharao/Documents/Documents - Kanderao’s MacBook Air/Python-Code/Gen-AI
Working from: /Users/kanderaolaxminarasimharao/Documents/Documents - Kanderao’s MacBook Air/Python-Code/Gen-AI/notebooks


In [7]:
from typing import Annotated, TypedDict, Sequence
import os

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# LangChain
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# LLM - choose based on what's in your .env
from langchain_openai import ChatOpenAI
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# from langchain_anthropic import ChatAnthropic

print("✅ All imports successful!")

✅ All imports successful!


In [8]:
print("📖 LESSON 1: Defining State")
print("="*80)

class ConversationState(TypedDict):
    """
    State with automatic message history management.
    
    Key Concept:
    - 'messages' stores conversation history
    - 'add_messages' reducer automatically appends new messages
    - No manual list management needed!
    """
    messages: Annotated[Sequence[BaseMessage], add_messages]

print("✓ State defined with add_messages reducer")
print("\nWhat add_messages does:")
print("  • Automatically appends new messages to history")
print("  • Handles deduplication")
print("  • Maintains order")
print("  • You just return new messages, it handles the rest!")

📖 LESSON 1: Defining State
✓ State defined with add_messages reducer

What add_messages does:
  • Automatically appends new messages to history
  • Handles deduplication
  • Maintains order
  • You just return new messages, it handles the rest!


In [5]:
print("📖 LESSON 2: Setting up LLM")
print("="*80)

# Auto-detect which API key is available
if os.getenv('OPENAI_API_KEY'):
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
    print("✓ Using OpenAI (GPT-3.5-turbo)")
    
elif os.getenv('NVIDIA_API_KEY'):
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct", temperature=0.7)
    print("✓ Using NVIDIA (Llama 3.1 70B)")
    
elif os.getenv('ANTHROPIC_API_KEY'):
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-3-sonnet-20240229", temperature=0.7)
    print("✓ Using Anthropic (Claude 3 Sonnet)")
    
else:
    raise ValueError("No LLM API key found! Add to .env file")

print(f"  Model: {llm.__class__.__name__}")

📖 LESSON 2: Setting up LLM
✓ Using OpenAI (GPT-3.5-turbo)
  Model: ChatOpenAI


In [9]:
print("📖 LESSON 3: Creating Agent Node")
print("="*80)

def call_model(state: ConversationState) -> ConversationState:
    """
    Agent node - the brain of our conversational agent.
    
    Flow:
    1. Receive state with message history
    2. Create prompt (system message + conversation history)
    3. Call LLM
    4. Return new message (add_messages appends automatically)
    
    Args:
        state: Current conversation state with messages
    
    Returns:
        Updated state with AI response
    """
    
    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful, friendly AI assistant. Be conversational and concise."),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    # Format messages
    formatted = prompt.format_messages(messages=state["messages"])
    
    # Call LLM
    response = llm.invoke(formatted)
    
    # Return new message (add_messages will append it to history)
    return {"messages": [response]}

print("✓ Agent node created")
print("\nImportant:")
print("  • We only return the NEW message")
print("  • add_messages reducer handles appending to history")
print("  • No manual list concatenation needed!")

📖 LESSON 3: Creating Agent Node
✓ Agent node created

Important:
  • We only return the NEW message
  • add_messages reducer handles appending to history
  • No manual list concatenation needed!


In [10]:
print("📖 LESSON 4: Building the Graph")
print("="*80)

# Create graph
workflow = StateGraph(ConversationState)

# Add agent node
workflow.add_node("agent", call_model)

# Set entry point (where execution starts)
workflow.set_entry_point("agent")

# Add edge from agent to END
workflow.add_edge("agent", END)

print("✓ Graph structure created")
print("\nGraph Flow:")
print("  START → [agent] → END")
print("\nExplanation:")
print("  • User message comes in")
print("  • Goes to 'agent' node")
print("  • Agent calls LLM")
print("  • Returns response")
print("  • Graph ends")

📖 LESSON 4: Building the Graph
✓ Graph structure created

Graph Flow:
  START → [agent] → END

Explanation:
  • User message comes in
  • Goes to 'agent' node
  • Agent calls LLM
  • Returns response
  • Graph ends


In [11]:
print("📖 LESSON 5: Adding Memory")
print("="*80)

# Create memory saver
memory = MemorySaver()

# Compile graph with checkpointing
app = workflow.compile(checkpointer=memory)

print("✓ Graph compiled with memory checkpointing")
print("\nWhat is MemorySaver?")
print("  • Stores conversation state in memory")
print("  • Maintains separate state per thread_id")
print("  • Allows conversation persistence")
print("  • Each thread_id = separate conversation")
print("\nWhy 'checkpointer'?")
print("  • Saves state after each node execution")
print("  • Allows resuming conversations")
print("  • Enables conversation replay")

📖 LESSON 5: Adding Memory
✓ Graph compiled with memory checkpointing

What is MemorySaver?
  • Stores conversation state in memory
  • Maintains separate state per thread_id
  • Allows conversation persistence
  • Each thread_id = separate conversation

Why 'checkpointer'?
  • Saves state after each node execution
  • Allows resuming conversations
  • Enables conversation replay


In [12]:
print("📖 LESSON 6: Creating Chat Interface")
print("="*80)

def chat(user_input: str, thread_id: str = "default") -> str:
    """
    Simple chat interface.
    
    Args:
        user_input: User's message
        thread_id: Conversation identifier (same ID = same conversation)
    
    Returns:
        AI's response as string
    
    Example:
        >>> chat("Hello!", thread_id="user_123")
        "Hi! How can I help you today?"
    """
    
    # Create config with thread ID
    config = {"configurable": {"thread_id": thread_id}}
    
    # Invoke graph
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    
    # Extract AI response (last message)
    ai_response = result["messages"][-1].content
    
    return ai_response

print("✓ Chat function ready")
print("\nUsage:")
print("  response = chat('Hello!', thread_id='alice')")
print("  response = chat('What's my name?', thread_id='alice')  # Remembers!")

📖 LESSON 6: Creating Chat Interface
✓ Chat function ready

Usage:
  response = chat('Hello!', thread_id='alice')
  response = chat('What's my name?', thread_id='alice')  # Remembers!


In [13]:
print("📖 LESSON 7: Testing Memory")
print("="*80)

thread = "demo_conversation"

# Conversation Turn 1
print("👤 User: Hi! My name is Alice and I'm learning LangGraph.")
response = chat("Hi! My name is Alice and I'm learning LangGraph.", thread)
print(f"🤖 AI: {response}\n")

# Conversation Turn 2 - Test memory
print("👤 User: What's my name?")
response = chat("What's my name?", thread)
print(f"🤖 AI: {response}\n")

# Conversation Turn 3 - Test context
print("👤 User: What am I learning about?")
response = chat("What am I learning about?", thread)
print(f"🤖 AI: {response}\n")

print("✅ SUCCESS! The AI remembered:")
print("  • Your name (Alice)")
print("  • What you're learning (LangGraph)")
print("  • Full conversation context")

📖 LESSON 7: Testing Memory
👤 User: Hi! My name is Alice and I'm learning LangGraph.
🤖 AI: Hi Alice! That's great to hear. LangGraph is an interesting subject. How can I assist you with your learning today?

👤 User: What's my name?
🤖 AI: Your name is Alice. How can I assist you further, Alice?

👤 User: What am I learning about?
🤖 AI: You are learning about LangGraph. It's a fascinating subject. Feel free to ask me any questions you have about it!

✅ SUCCESS! The AI remembered:
  • Your name (Alice)
  • What you're learning (LangGraph)
  • Full conversation context


In [14]:
print("📖 LESSON 8: Multiple Conversation Threads")
print("="*80)

# Thread 1 - Alice
print("💬 Conversation Thread: ALICE")
print("-"*80)
chat("My favorite color is blue.", "alice")
print("Alice: My favorite color is blue.\n")

# Thread 2 - Bob  
print("💬 Conversation Thread: BOB")
print("-"*80)
chat("My favorite color is red.", "bob")
print("Bob: My favorite color is red.\n")

# Back to Alice - test memory isolation
print("💬 Back to Thread: ALICE")
print("-"*80)
print("Alice: What's my favorite color?")
response = chat("What's my favorite color?", "alice")
print(f"AI: {response}")
print("✓ Correctly remembered: BLUE\n")

# Back to Bob - test memory isolation
print("💬 Back to Thread: BOB")
print("-"*80)
print("Bob: What's my favorite color?")
response = chat("What's my favorite color?", "bob")
print(f"AI: {response}")
print("✓ Correctly remembered: RED\n")

print("✅ SUCCESS! Each thread has SEPARATE memory:")
print("  • Alice's thread remembers 'blue'")
print("  • Bob's thread remembers 'red'")
print("  • No cross-contamination!")

📖 LESSON 8: Multiple Conversation Threads
💬 Conversation Thread: ALICE
--------------------------------------------------------------------------------
Alice: My favorite color is blue.

💬 Conversation Thread: BOB
--------------------------------------------------------------------------------
Bob: My favorite color is red.

💬 Back to Thread: ALICE
--------------------------------------------------------------------------------
Alice: What's my favorite color?
AI: Your favorite color is blue! It's a lovely choice.
✓ Correctly remembered: BLUE

💬 Back to Thread: BOB
--------------------------------------------------------------------------------
Bob: What's my favorite color?
AI: Your favorite color is red! If there's anything else you'd like to share or ask about, feel free to let me know.
✓ Correctly remembered: RED

✅ SUCCESS! Each thread has SEPARATE memory:
  • Alice's thread remembers 'blue'
  • Bob's thread remembers 'red'
  • No cross-contamination!


In [15]:
print("📖 LESSON 9: Viewing Conversation History")
print("="*80)

def get_history(thread_id: str):
    """Get full conversation history for a thread"""
    config = {"configurable": {"thread_id": thread_id}}
    state = app.get_state(config)
    return state.values.get("messages", [])

# Show Alice's full history
print("📜 Alice's Full Conversation History:")
print("-"*80)

history = get_history("alice")
for i, msg in enumerate(history, 1):
    if isinstance(msg, HumanMessage):
        print(f"{i}. 👤 Alice: {msg.content}")
    elif isinstance(msg, AIMessage):
        print(f"{i}. 🤖 AI: {msg.content}")

print("\n✓ Full conversation accessible via get_history()")

📖 LESSON 9: Viewing Conversation History
📜 Alice's Full Conversation History:
--------------------------------------------------------------------------------
1. 👤 Alice: My favorite color is blue.
2. 🤖 AI: Blue is a great color! It's often associated with calmness and serenity. Do you have a specific shade of blue that you like the most?
3. 👤 Alice: What's my favorite color?
4. 🤖 AI: Your favorite color is blue! It's a lovely choice.

✓ Full conversation accessible via get_history()


In [16]:
print("📖 LESSON 10: Clearing Conversation History")
print("="*80)

def clear_history(thread_id: str):
    """Clear a conversation's history"""
    config = {"configurable": {"thread_id": thread_id}}
    app.update_state(config, {"messages": []})
    print(f"✓ Cleared conversation: {thread_id}")

# Clear Alice's conversation
clear_history("alice")

# Test - should NOT remember anymore
print("\nAlice: What's my favorite color?")
response = chat("What's my favorite color?", "alice")
print(f"AI: {response}")
print("\n✅ Memory successfully cleared - AI doesn't remember 'blue' anymore")

📖 LESSON 10: Clearing Conversation History
✓ Cleared conversation: alice

Alice: What's my favorite color?
AI: Your favorite color is blue!

✅ Memory successfully cleared - AI doesn't remember 'blue' anymore


In [17]:
print("📖 LESSON 11: Interactive Chat Widget (Jupyter)")
print("="*80)

from IPython.display import display
import ipywidgets as widgets

# Create UI widgets
output_area = widgets.Output()
conversation_display = widgets.HTML(value="<h3>💬 Conversation</h3>")

user_input = widgets.Text(
    placeholder='Type your message here...',
    description='You:',
    layout=widgets.Layout(width='70%')
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    icon='paper-plane'
)

clear_button = widgets.Button(
    description='Clear',
    button_style='warning',
    icon='trash'
)

# State
thread_id = "jupyter_interactive"
conversation_html = []

def on_send(b):
    """Handle send button click"""
    global conversation_html
    
    if not user_input.value.strip():
        return
    
    # Get user message
    user_msg = user_input.value
    
    # Get AI response
    ai_response = chat(user_msg, thread_id)
    
    # Update display
    conversation_html.append(
        f"<p style='background-color: #e3f2fd; padding: 10px; border-radius: 5px;'>"
        f"<b>👤 You:</b> {user_msg}</p>"
    )
    conversation_html.append(
        f"<p style='background-color: #f1f8e9; padding: 10px; border-radius: 5px;'>"
        f"<b>🤖 AI:</b> {ai_response}</p>"
    )
    
    conversation_display.value = "<h3>💬 Conversation</h3>" + "".join(conversation_html)
    
    # Clear input
    user_input.value = ""

def on_clear(b):
    """Handle clear button click"""
    global conversation_html
    conversation_html = []
    conversation_display.value = "<h3>💬 Conversation</h3>"
    clear_history(thread_id)
    print("✓ Conversation cleared")

# Connect buttons
send_button.on_click(on_send)
clear_button.on_click(on_clear)

# Display UI
print("✅ Interactive chat widget created!\n")

display(widgets.VBox([
    conversation_display,
    widgets.HBox([user_input, send_button, clear_button])
]))

print("\n💡 Try chatting!")
print("  • Type a message and click Send")
print("  • The AI will remember context")
print("  • Click Clear to reset conversation")

📖 LESSON 11: Interactive Chat Widget (Jupyter)
✅ Interactive chat widget created!




💡 Try chatting!
  • Type a message and click Send
  • The AI will remember context
  • Click Clear to reset conversation


In [18]:
print("="*80)
print("🎉 TUTORIAL 1 COMPLETE!")
print("="*80)

print("\n✅ What You Learned:")
print("  1. ✓ Define State with TypedDict and add_messages")
print("  2. ✓ Create agent nodes that process state")
print("  3. ✓ Build LangGraph workflows")
print("  4. ✓ Compile with memory checkpointing")
print("  5. ✓ Manage multiple conversation threads")
print("  6. ✓ View and clear conversation history")
print("  7. ✓ Create interactive chat interfaces")

print("\n🎯 Key Concepts Mastered:")
print("  • State: Data flowing through the graph")
print("  • Nodes: Functions that transform state")
print("  • Edges: Connections between nodes")
print("  • Checkpointer: Memory persistence system")
print("  • Thread ID: Conversation identifier")
print("  • add_messages: Automatic message history management")

print("\n💡 Try These Exercises:")
print("  1. Add a custom system prompt")
print("  2. Add a 'summarize' command that summarizes the conversation")
print("  3. Track user sentiment across messages")
print("  4. Add a 'search knowledge base' feature")
print("  5. Create multiple personas (teacher, comedian, etc.)")

print("\n📚 Ready for Tutorial 2?")
print("  Next: Academic Task Learning Agent (ATLAS)")
print("  • Multi-agent orchestration")
print("  • Complex state management")
print("  • Real-world planning system")

print("\n" + "="*80)

🎉 TUTORIAL 1 COMPLETE!

✅ What You Learned:
  1. ✓ Define State with TypedDict and add_messages
  2. ✓ Create agent nodes that process state
  3. ✓ Build LangGraph workflows
  4. ✓ Compile with memory checkpointing
  5. ✓ Manage multiple conversation threads
  6. ✓ View and clear conversation history
  7. ✓ Create interactive chat interfaces

🎯 Key Concepts Mastered:
  • State: Data flowing through the graph
  • Nodes: Functions that transform state
  • Edges: Connections between nodes
  • Checkpointer: Memory persistence system
  • Thread ID: Conversation identifier
  • add_messages: Automatic message history management

💡 Try These Exercises:
  1. Add a custom system prompt
  2. Add a 'summarize' command that summarizes the conversation
  3. Track user sentiment across messages
  4. Add a 'search knowledge base' feature
  5. Create multiple personas (teacher, comedian, etc.)

📚 Ready for Tutorial 2?
  Next: Academic Task Learning Agent (ATLAS)
  • Multi-agent orchestration
  • Comple

In [19]:
# This works immediately - no installation needed
import sqlite3

print(f"✓ SQLite version: {sqlite3.sqlite_version}")
# Output: ✓ SQLite version: 3.45.1 (or similar)

✓ SQLite version: 3.51.0


In [20]:
!git init

Initialized empty Git repository in /Users/kanderaolaxminarasimharao/Documents/Documents - Kanderao’s MacBook Air/Python-Code/Gen-AI/notebooks/.git/


In [21]:
!git add SimpleAgent.ipynb

In [22]:
!git commit -m "Add Jupyter notebook file"

[main (root-commit) 651f6f2] Add Jupyter notebook file
 1 file changed, 1011 insertions(+)
 create mode 100644 SimpleAgent.ipynb


In [23]:
!git remote add origin https://github.com/klnsuman/MyAgents.git


In [ ]:
!git push -u origin main

Username for 'https://github.com': 